In [74]:
import pandas as pd
import hashlib
import re 
from pathlib import Path

df = pd.read_csv('statement.csv', skiprows=1)
print(df.head())

In [75]:
print(df.dtypes)

In [76]:
def load_raw(file_path):
    df = pd.read_csv(file_path, skiprows=1)
    return df   

In [77]:
def rename_drop_columns(df):
    df = df.copy()
    df.columns = ['card_number', 'transaction_type', 'date', 'amount', 'description']
    df= df.drop(columns=['card_number'])
    return df 

In [78]:
def parse_date(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], format="%Y%m%d")
    return df

In [79]:
def clean_description(df):
    df = df.copy()
    df['description'] = df['description'].str.replace("\xa0", " ", regex=False)
    df['description'] = df['description'].str.replace(r"\s+", " ", regex=True)
    df['description'] = df['description'].str.strip()
    return df

In [80]:
def extract_transaction_codes(df):
    df = df.copy()
    df["transaction_code"] = df["description"].str.extract(r'^\[([A-Z]+)\]')
    df["description"] = df["description"].str.replace(r'^\[[A-Z]+\]\s*', '', regex=True)
    return df

In [81]:
def generate_fingerprint(date, amount, description):
    raw = f"{str(date)}{str(amount)}{str(description)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def add_fingerprint(df):
    df = df.copy()
    df['fingerprint'] = df.apply(
    lambda row: generate_fingerprint(row["date"], row["amount"], row["description"]),
    axis=1
    )
    return df

In [82]:
def parse_bmo_csv(file_path):
    """
    Accepts a path to a csv file and returns a dataframe of clean 
    transaction history, ready for database insertion.
    """ 
    
    if not Path(file_path).exists():
        raise FileNotFoundError(f"csv file not found: {file_path}")
    df = load_raw(file_path)
    df = rename_drop_columns(df)
    df = parse_date(df)
    df = clean_description(df)
    df = extract_transaction_codes(df)
    df = add_fingerprint(df)
    return df

In [83]:
# test the result 
df = parse_bmo_csv('statement.csv')
print(df.head())